# ソースコードを測って読み解く（実践編）

複数の Python ソースを解析して**コードメトリクス**（行数・循環的複雑度・関数数など）のデータセットを作り、規模と複雑さの関係を調べ、KMeans で似たコードを群に分ける総合演習です。**プログラム解析 → 前処理 → 機械学習（教師なし）** を 1 本のパイプラインとしてつなぎます。

- 解析対象: 標準ライブラリのモジュール（ネットワーク不要で自己完結）
- 使うライブラリ: `ast`（標準）, `radon`, `pandas`, `scikit-learn`, `matplotlib`
- 対応するページ: `learn/practice-code-metrics.html`

表示される件数や数値は Python のバージョンによって多少変わります。


## 準備: ライブラリのインストール
Colab では最初にこのセルを実行します。

In [ ]:
!pip install radon scikit-learn pandas matplotlib

## Step 1: 解析するソースを集める

標準ライブラリの置き場所は `sysconfig.get_paths()["stdlib"]` で分かります。そこにある単一ファイルのモジュール（`*.py`）を集め、名前が `_` で始まる内部用のものは除きます。

In [ ]:
import sysconfig, glob, os

# 標準ライブラリの置き場所を調べ、単一ファイルのモジュールを集める
stdlib_dir = sysconfig.get_paths()["stdlib"]
paths = sorted(glob.glob(os.path.join(stdlib_dir, "*.py")))
# 名前が "_" で始まる内部用モジュールは除く
paths = [p for p in paths if not os.path.basename(p).startswith("_")]

print("解析対象:", stdlib_dir)
print("候補ファイル数:", len(paths))
print("例:", [os.path.basename(p) for p in paths[:6]])

## Step 2: ast と radon で計測する

各ファイルから次を取り出します。

- **行数系**: `radon.raw.analyze` が返す `loc`（総行数）・`sloc`（実行行数）・`comments`（コメント行数）
- **関数数・クラス数**: `ast.parse` で構文木を作り、`ast.walk` で `FunctionDef` と `ClassDef` を数える
- **循環的複雑度**: `radon.complexity.cc_visit` が返す関数ごとの複雑度から、最大 `max_cc` と平均 `mean_cc`

**循環的複雑度**は「判定ノードの数 + 1」で数えます（if・elif・for・while・and・or・except・三項演算子・内包表記の if などの分岐 1 つにつき +1、分岐がなければ 1）。グラフ理論の定義 M = E - N + 2P の実用的な等価表現です。

In [ ]:
import ast
from radon.raw import analyze
from radon.complexity import cc_visit

def measure(path):
    name = os.path.splitext(os.path.basename(path))[0]
    src = open(path, encoding="utf-8", errors="ignore").read()
    try:
        raw = analyze(src)        # 行数系（loc, sloc, comments ...）
        tree = ast.parse(src)     # 構文木
        blocks = cc_visit(src)    # 関数・メソッドごとの循環的複雑度
    except (SyntaxError, UnicodeError):
        return None
    funcs = [n for n in ast.walk(tree)
             if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
    classes = [n for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
    ccs = [b.complexity for b in blocks]
    return {
        "module": name,
        "loc": raw.loc, "sloc": raw.sloc, "comments": raw.comments,
        "functions": len(funcs), "classes": len(classes),
        "max_cc": max(ccs) if ccs else 0,
        "mean_cc": round(sum(ccs) / len(ccs), 2) if ccs else 0.0,
    }

print(measure(os.path.join(stdlib_dir, "csv.py")))

`cc_visit` は関数ごとの複雑度を持つので、「どの関数がいちばん入り組んでいるか」も調べられます。

In [ ]:
src = open(os.path.join(stdlib_dir, "tokenize.py"), encoding="utf-8").read()
for b in sorted(cc_visit(src), key=lambda b: -b.complexity)[:3]:
    print(f"{b.name:<16} cc={b.complexity}")

## Step 3: 表に整え、外れ値を見て、標準化する

すべてのファイルに `measure` を適用し、`None` を除いて pandas の表にします。中身が数行の「薄いラッパ」は `sloc` が極端に小さく分析のノイズになりやすいので、`sloc` が 40 行未満のものを外れ値として除きます。

In [ ]:
import pandas as pd

# すべてのファイルを計測し、失敗した（None の）ものは除く
records = [m for p in paths if (m := measure(p)) is not None]
df = pd.DataFrame(records)
print("解析できたモジュール数:", len(df))

# 薄いラッパなど極端に小さいファイルは外れ値として除く
df = df[df["sloc"] >= 40].reset_index(drop=True)
print("前処理後:", len(df))

features = ["sloc", "comments", "functions", "classes", "max_cc", "mean_cc"]
print(df[features].describe().round(1))

`sloc` は数百〜数千、`mean_cc` は 1 桁と桁が違います。距離を測る前に `StandardScaler` で各列を「平均 0・標準偏差 1」にそろえ、すべての指標を対等にします。

In [ ]:
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df[features])
print("標準化後の平均:", X.mean(axis=0).round(2))
print("標準化後の標準偏差:", X.std(axis=0).round(2))

## Step 4: 規模と複雑さの関係を調べる

クラスタリングの前に、指標どうしの相関を確かめます。相関係数はマイナス 1 から 1 の値をとり、1 に近いほど一方が増えると他方も増える関係が強いことを表します。

In [ ]:
corr = df[features].corr()
print("規模(sloc) と 最大複雑度(max_cc):", round(corr.loc["sloc", "max_cc"], 3))
print("規模(sloc) と 関数数(functions):", round(corr.loc["sloc", "functions"], 3))
print("関数数 と クラス数:", round(corr.loc["functions", "classes"], 3))

規模と関数数は強く相関しますが、規模と最大複雑度の相関はほどほどです。「小さいのに一部だけ複雑」なファイルが存在することを示唆します。散布図で確かめます。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter(df["sloc"], df["max_cc"], s=18, alpha=0.6)
plt.xlabel("sloc (実行行数)")
plt.ylabel("max_cc (最大循環的複雑度)")
plt.title("規模 vs 複雑さ")
plt.tight_layout()
plt.show()

## Step 5: KMeans で似たコードを群化する

標準化した 6 次元空間で、`KMeans` を使ってファイルを `k=4` 個の群に分けます。正解ラベルはないので、分けたあとで**各群の平均**を見て性格を解釈します。

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X)

print(df["cluster"].value_counts().sort_index())
print(df.groupby("cluster")[features].mean().round(1))

各群のメンバーを覗いてみます。

In [ ]:
for c in sorted(df["cluster"].unique()):
    names = df[df["cluster"] == c]["module"].head(5).tolist()
    print(f"cluster {c}:", ", ".join(names))

6 次元のままでは見られないので、`PCA` で 2 次元に落として散布図にします。PCA は情報の損失を抑えて次元を圧縮する手法です。

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pts = pca.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(pts[:, 0], pts[:, 1], c=df["cluster"], cmap="viridis", s=22, alpha=0.75)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("KMeans クラスタ（PCAで2次元に投影）")
plt.tight_layout()
plt.show()

print("PCAが説明する分散の割合:", pca.explained_variance_ratio_.round(3))

## Step 6: 結果を読み、限界を知る

標準ライブラリのモジュールは、規模と複雑さでおおむね「小さく素直」「小さいのに局所的に複雑」「中規模でクラス構成的」「大規模で機能豊富」の 4 群に分かれました。

大切なのは、**複雑度が高いこと＝悪いこととは限らない**という点です。字句解析やパーサ（`tokenize` の `_tokenize` など）は本質的に多くの場合分けを必要とします。メトリクスは「ここに注意を向ける価値がある」という目印であって、良し悪しの判定そのものではありません。

**限界**: メトリクスは表面的な形だけを測り、設計の良さや正しさは測れません。`k` の値・特徴量・外れ値の基準で結果は変わり、唯一の正解はありません。最後は人間がコードを読んで確かめます。

## 発展課題

1. **別のコードを対象にする**: `pip` で入れた小さな公開パッケージのソース（`パッケージ名.__path__`）に差し替える。
2. **特徴量を足す**: `radon` の `h_visit`（Halstead 指標）や、コメント行と実行行の比を加える。
3. **クラスタ数を変える**: `k` を 3 や 6 にし、`km.inertia_` を記録してエルボー法で「ほどよい `k`」を探す。
4. **保守性指数を計算する**: `radon.metrics.mi_visit` で Maintainability Index を出し、群ごとに比べる。
